# Shared Silver Player Participation and Weekly Facts

## tl;dr

This notebook creates the first shared Silver datasets for the 2023–2025 scope. The stats-plus-identified-snaps union contains 79,708 player-game observations: 56,930 appear in both sources, 52 appear only in weekly stats, and 22,726 appear only in mapped snap counts.

Exact PFR-to-GSIS matching resolves 79,656 snap rows. Another 111 snap rows across 11 PFR identifiers remain explicit identity exceptions and do not enter canonical GSIS-grained output. Of the mapped snap rows, 79,651 record at least one snap and five confirm zero total snaps. Participation remains null for the 52 stats-only observations because no mapped snap evidence exists.

Both `silver_player_game_participation` and `silver_player_week` contain 79,708 rows and pass their declared grain, schedule, source-reconciliation, and Parquet round-trip checks. The contextual WR slice contains 8,881 player-weeks, matching the preceding validation notebook. Six season-partitioned Silver Parquet files are written.

## Context & Methods

### Scope

- Seasons: 2023–2025
- Output population: all identified players with weekly-stat or mapped snap evidence
- Primary validation lens: wide receivers
- Input format: persisted Bronze Parquet
- Output format: season-partitioned Silver Parquet
- Environment: `sports_dev_env` with pandas

This notebook implements the identity, participation, position, and grain contracts established in `01_player_identity_participation_grain_validation.ipynb`. It produces shared football facts rather than a WR-only foundation, so later RB and QB work can reuse the same tables.

### Key Assumptions

- GSIS is the canonical `player_id`; player names are never join keys.
- PFR snap identifiers enter canonical output only through the one-to-one player-master bridge.
- The row universe is the union of identified weekly-stat and mapped snap observations.
- `participated_game` requires at least one positive offensive, defensive, or special-teams snap.
- `offensive_participant` requires at least one positive offensive snap.
- Missing snap evidence remains null rather than becoming zero or false.
- Weekly roster data enriches observed facts but does not generate rows for every roster state.
- Detailed playoff rounds remain in `game_type`; `season_type` normalizes them to `POST`.
- Fantasy points and source-provided opportunity shares remain outside these scoring-neutral Silver facts.

## Data

### 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

SEASONS = [2023, 2024, 2025]
POSTSEASON_GAME_TYPES = {"WC", "DIV", "CON", "SB"}
GAME_GRAIN = ["player_id", "game_id"]
WEEKLY_GRAIN = ["player_id", "season", "week", "team"]
SOURCE_GAME_KEY = ["player_id", "game_id", "team"]

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "ROADMAP.md").exists():
            PROJECT_ROOT = parent
            break
    else:
        raise RuntimeError("Could not locate the repository root.")

BRONZE_DIR = PROJECT_ROOT / "data/bronze"
SILVER_DIR = PROJECT_ROOT / "data/silver"
GAME_PARTICIPATION_DIR = SILVER_DIR / "player_game_participation"
PLAYER_WEEK_DIR = SILVER_DIR / "player_week"

PROJECT_ROOT

PosixPath('/Users/dtwice/Development/dev_sports/nfl/fantasy-football/nfl-dream-lab-app')

### 2. Define the scoring-neutral weekly statistics

These fields are source-recorded football facts needed across WR, RB, and QB work. Negative yardage is valid football data and is preserved. Source fantasy points and precomputed shares are intentionally excluded.

In [2]:
STAT_COLUMN_RENAMES = {
    "completions": "passing_completions",
    "attempts": "passing_attempts",
    "passing_yards": "passing_yards",
    "passing_tds": "passing_touchdowns",
    "passing_interceptions": "passing_interceptions",
    "sacks_suffered": "sacks_suffered",
    "passing_first_downs": "passing_first_downs",
    "passing_2pt_conversions": "passing_2pt_conversions",
    "carries": "rushing_attempts",
    "rushing_yards": "rushing_yards",
    "rushing_tds": "rushing_touchdowns",
    "rushing_first_downs": "rushing_first_downs",
    "rushing_2pt_conversions": "rushing_2pt_conversions",
    "targets": "receiving_targets",
    "receptions": "receptions",
    "receiving_yards": "receiving_yards",
    "receiving_tds": "receiving_touchdowns",
    "receiving_air_yards": "receiving_air_yards",
    "receiving_yards_after_catch": "receiving_yards_after_catch",
    "receiving_first_downs": "receiving_first_downs",
    "receiving_2pt_conversions": "receiving_2pt_conversions",
    "fumbles_total": "fumbles",
    "fumbles_lost_total": "fumbles_lost",
    "special_teams_tds": "special_teams_touchdowns",
}

SOURCE_STAT_COLUMNS = list(STAT_COLUMN_RENAMES)
SILVER_STAT_COLUMNS = list(STAT_COLUMN_RENAMES.values())

len(SILVER_STAT_COLUMNS)

24

### 3. Load the persisted Bronze sources

In [3]:
players_path = BRONZE_DIR / "players/players.parquet"
schedule_paths = sorted((BRONZE_DIR / "schedules").glob("*.parquet"))
player_stats_paths = sorted((BRONZE_DIR / "player_stats_weekly").glob("*.parquet"))
roster_paths = sorted((BRONZE_DIR / "rosters_weekly").glob("*.parquet"))
snap_paths = sorted((BRONZE_DIR / "snap_counts").glob("*.parquet"))

assert players_path.exists()
assert len(schedule_paths) == len(SEASONS)
assert len(player_stats_paths) == len(SEASONS)
assert len(roster_paths) == len(SEASONS)
assert len(snap_paths) == len(SEASONS)

players = pd.read_parquet(players_path)
schedules = pd.concat(
    [pd.read_parquet(path) for path in schedule_paths],
    ignore_index=True,
)
player_stats = pd.concat(
    [pd.read_parquet(path) for path in player_stats_paths],
    ignore_index=True,
)
rosters = pd.concat(
    [pd.read_parquet(path) for path in roster_paths],
    ignore_index=True,
)
snap_counts = pd.concat(
    [pd.read_parquet(path) for path in snap_paths],
    ignore_index=True,
)

required_stat_columns = {
    "player_id", "player_display_name", "position", "position_group",
    "season", "week", "season_type", "game_id", "team", "opponent_team",
    *SOURCE_STAT_COLUMNS,
}
assert required_stat_columns.issubset(player_stats.columns)

In [4]:
input_summary = pd.DataFrame(
    [
        ["players", len(players), len(players.columns), 1],
        ["schedules", len(schedules), len(schedules.columns), len(schedule_paths)],
        ["player_stats_weekly", len(player_stats), len(player_stats.columns), len(player_stats_paths)],
        ["rosters_weekly", len(rosters), len(rosters.columns), len(roster_paths)],
        ["snap_counts", len(snap_counts), len(snap_counts.columns), len(snap_paths)],
    ],
    columns=["dataset", "rows", "columns", "files"],
)

for source in [schedules, player_stats, rosters, snap_counts]:
    assert set(source["season"].dropna().unique()) == set(SEASONS)

input_summary

,dataset,rows,columns,files
0,players,24828,39,1
1,schedules,855,46,3
2,player_stats_weekly,57048,150,3
3,rosters_weekly,139083,36,3
4,snap_counts,79767,16,3


## Results

### 4. Create canonical team-game context

Each schedule row becomes one row per participating team. This table is authoritative for season, week, opponent, and game type.

In [5]:
home_team_games = schedules[
    ["game_id", "season", "week", "game_type", "home_team", "away_team"]
].rename(columns={"home_team": "team", "away_team": "opponent_team"})

away_team_games = schedules[
    ["game_id", "season", "week", "game_type", "away_team", "home_team"]
].rename(columns={"away_team": "team", "home_team": "opponent_team"})

schedule_team_games = pd.concat(
    [home_team_games, away_team_games],
    ignore_index=True,
)
schedule_team_games["season_type"] = schedule_team_games["game_type"].map(
    lambda game_type: "POST" if game_type in POSTSEASON_GAME_TYPES else game_type
)

assert not schedule_team_games.duplicated(["game_id", "team"]).any()
assert not schedule_team_games.duplicated(["season", "week", "team"]).any()
assert schedule_team_games[["game_id", "season", "week", "team", "opponent_team"]].notna().all().all()
assert set(schedule_team_games["season_type"].unique()) == {"REG", "POST"}

(
    schedule_team_games.groupby(["season", "season_type"], dropna=False)
    .agg(
        team_games=("game_id", "size"),
        teams=("team", "nunique"),
        min_week=("week", "min"),
        max_week=("week", "max"),
    )
    .reset_index()
)

,season,season_type,team_games,teams,min_week,max_week
0,2023,POST,26,14,19,22
1,2023,REG,544,32,1,18
2,2024,POST,26,14,19,22
3,2024,REG,544,32,1,18
4,2025,POST,26,14,19,22
5,2025,REG,544,32,1,18


### 5. Prepare exact player identity and source exceptions

The player master supplies canonical display metadata and the one-to-one PFR bridge. Null weekly-stat IDs and unmatched snap PFR IDs remain explicit exceptions.

In [6]:
player_dimension = players[
    ["gsis_id", "display_name", "position", "position_group"]
].rename(
    columns={
        "gsis_id": "player_id",
        "position": "master_position",
        "position_group": "master_position_group",
    }
)

master_pfr_bridge = players.loc[
    players["pfr_id"].notna(),
    ["pfr_id", "gsis_id"],
].rename(columns={"gsis_id": "player_id"})

assert player_dimension["player_id"].notna().all()
assert not player_dimension["player_id"].duplicated().any()
assert not master_pfr_bridge["pfr_id"].duplicated().any()
assert not master_pfr_bridge["player_id"].duplicated().any()

null_stat_identity_exceptions = player_stats.loc[
    player_stats["player_id"].isna(),
    ["season", "week", "season_type", "game_id", "team", *SOURCE_STAT_COLUMNS],
].copy()

null_stat_identity_exceptions["has_selected_stat_value"] = (
    null_stat_identity_exceptions[SOURCE_STAT_COLUMNS].fillna(0).ne(0).any(axis=1)
)

assert not null_stat_identity_exceptions["has_selected_stat_value"].any()

print(f"Weekly-stat rows without player_id: {len(null_stat_identity_exceptions):,}")
null_stat_identity_exceptions[
    ["season", "week", "season_type", "game_id", "team", "has_selected_stat_value"]
].head(12)

Weekly-stat rows without player_id: 66


,season,week,season_type,game_id,team,has_selected_stat_value
1057,2023,1,REG,2023_01_DET_KC,DET,False
2073,2023,2,REG,2023_02_MIN_PHI,PHI,False
3110,2023,3,REG,2023_03_NYG_SF,SF,False
4184,2023,4,REG,2023_04_DET_GB,DET,False
5091,2023,5,REG,2023_05_CHI_WAS,CHI,False
6070,2023,6,REG,2023_06_DEN_KC,DEN,False
6910,2023,7,REG,2023_07_JAX_NO,JAX,False
7962,2023,8,REG,2023_08_TB_BUF,TB,False
8908,2023,9,REG,2023_09_TEN_PIT,PIT,False
9833,2023,10,REG,2023_10_CAR_CHI,CHI,False


### 6. Prepare snap participation

All mapped snap rows remain at their source player-game-team grain. Snap counts establish nullable participation flags before the stats union is built.

In [7]:
snap_identity = snap_counts.merge(
    master_pfr_bridge,
    left_on="pfr_player_id",
    right_on="pfr_id",
    how="left",
    validate="many_to_one",
    indicator="identity_join",
)

unmatched_snap_identity_exceptions = snap_identity.loc[
    snap_identity["identity_join"].ne("both")
].copy()

mapped_snap_source = snap_identity.loc[
    snap_identity["identity_join"].eq("both")
].copy()

assert not mapped_snap_source.duplicated(["player_id", "game_id", "team"]).any()
assert mapped_snap_source[["player_id", "game_id", "team"]].notna().all().all()

snap_identity_summary = (
    unmatched_snap_identity_exceptions.groupby("position", dropna=False)
    .agg(rows=("pfr_player_id", "size"), pfr_ids=("pfr_player_id", "nunique"))
    .sort_values("rows", ascending=False)
    .reset_index()
)

print(
    f"Mapped snap rows: {len(mapped_snap_source):,}; "
    f"unmatched rows: {len(unmatched_snap_identity_exceptions):,}; "
    f"unmatched PFR IDs: {unmatched_snap_identity_exceptions['pfr_player_id'].nunique():,}"
)
snap_identity_summary

Mapped snap rows: 79,656; unmatched rows: 111; unmatched PFR IDs: 11


,position,rows,pfr_ids
0,TE,28,2
1,G,23,2
2,T,19,1
3,WR,14,1
4,DL,12,1
5,RB,7,1
6,CB,5,2
7,DT,2,1
8,QB,1,1


In [8]:
prepared_snaps = mapped_snap_source[
    [
        "player_id",
        "game_id",
        "team",
        "player",
        "pfr_player_id",
        "position",
        "offense_snaps",
        "defense_snaps",
        "st_snaps",
        "offense_pct",
        "defense_pct",
        "st_pct",
    ]
].rename(
    columns={
        "player": "snap_player_name",
        "position": "snap_position",
        "st_snaps": "special_teams_snaps",
        "offense_pct": "offense_snap_pct",
        "defense_pct": "defense_snap_pct",
        "st_pct": "special_teams_snap_pct",
    }
)
prepared_snaps["has_snap_record"] = True
prepared_snaps["participated_game"] = (
    prepared_snaps[["offense_snaps", "defense_snaps", "special_teams_snaps"]]
    .gt(0)
    .any(axis=1)
    .astype("boolean")
)
prepared_snaps["offensive_participant"] = (
    prepared_snaps["offense_snaps"].gt(0).astype("boolean")
)

for column in ["offense_snaps", "defense_snaps", "special_teams_snaps"]:
    assert prepared_snaps[column].ge(0).all()
    prepared_snaps[column] = prepared_snaps[column].astype("Int64")

for column in ["offense_snap_pct", "defense_snap_pct", "special_teams_snap_pct"]:
    assert prepared_snaps[column].between(0, 1, inclusive="both").all()
    prepared_snaps[column] = prepared_snaps[column].astype("Float64")

participation_evidence_summary = pd.DataFrame(
    {
        "evidence": [
            "mapped snap rows",
            "at least one total snap",
            "at least one offensive snap",
            "all snap counts equal zero",
        ],
        "rows": [
            len(prepared_snaps),
            prepared_snaps["participated_game"].sum(),
            prepared_snaps["offensive_participant"].sum(),
            prepared_snaps["participated_game"].eq(False).sum(),
        ],
    }
)
participation_evidence_summary

,evidence,rows
0,mapped snap rows,79656
1,at least one total snap,79651
2,at least one offensive snap,31596
3,all snap counts equal zero,5


### 7. Prepare weekly-stat observations

Only identified rows enter canonical facts. Source names and positions remain available independently from master, roster, and snap metadata.

In [9]:
identified_player_stats = player_stats.loc[player_stats["player_id"].notna()].copy()

assert identified_player_stats["player_id"].isin(player_dimension["player_id"]).all()
assert not identified_player_stats.duplicated(SOURCE_GAME_KEY).any()
assert identified_player_stats[SOURCE_GAME_KEY].notna().all().all()

prepared_stats = identified_player_stats[
    [
        "player_id",
        "game_id",
        "team",
        "player_display_name",
        "position",
        "position_group",
        *SOURCE_STAT_COLUMNS,
    ]
].rename(
    columns={
        "player_display_name": "stats_player_name",
        "position": "stats_position",
        "position_group": "stats_position_group",
        **STAT_COLUMN_RENAMES,
    }
)
prepared_stats["has_stat_record"] = True

for column in SILVER_STAT_COLUMNS:
    prepared_stats[column] = prepared_stats[column].astype("Int64")

prepared_stats.head()

,player_id,game_id,team,stats_player_name,stats_position,stats_position_group,passing_completions,passing_attempts,passing_yards,passing_touchdowns,passing_interceptions,sacks_suffered,passing_first_downs,passing_2pt_conversions,rushing_attempts,rushing_yards,rushing_touchdowns,rushing_first_downs,rushing_2pt_conversions,receiving_targets,receptions,receiving_yards,receiving_touchdowns,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_2pt_conversions,fumbles,fumbles_lost,special_teams_touchdowns,has_stat_record
0,00-0023459,2023_01_BUF_NYJ,NYJ,Aaron Rodgers,QB,QB,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,True
1,00-0023853,2023_01_ARI_WAS,ARI,Matt Prater,K,SPEC,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,True
2,00-0025565,2023_01_TEN_NO,TEN,Nick Folk,K,SPEC,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,True
3,00-0026190,2023_01_CAR_ATL,ATL,Calais Campbell,DE,DL,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,True
4,00-0026498,2023_01_LA_SEA,LA,Matthew Stafford,QB,QB,24,38,334,0,0,0,17,0,3,11,0,1,0,0,0,0,0,0,0,0,0,0,0,0,True


### 8. Validate source-to-schedule relationships

Schedule agreement is checked before source context is discarded. A mismatch here would block construction because team, season, week, and opponent would be ambiguous.

In [10]:
stat_schedule_check = identified_player_stats.merge(
    schedule_team_games,
    on=["game_id", "team"],
    how="left",
    validate="many_to_one",
    suffixes=("_source", "_schedule"),
    indicator="schedule_join",
)

snap_schedule_check = mapped_snap_source.merge(
    schedule_team_games,
    on=["game_id", "team"],
    how="left",
    validate="many_to_one",
    suffixes=("_source", "_schedule"),
    indicator="schedule_join",
)

source_schedule_summary = pd.DataFrame(
    [
        {
            "source": "identified weekly stats",
            "rows": len(stat_schedule_check),
            "missing schedule rows": stat_schedule_check["schedule_join"].ne("both").sum(),
            "season mismatches": stat_schedule_check["season_source"].ne(stat_schedule_check["season_schedule"]).sum(),
            "week mismatches": stat_schedule_check["week_source"].ne(stat_schedule_check["week_schedule"]).sum(),
            "opponent mismatches": stat_schedule_check["opponent_team_source"].ne(stat_schedule_check["opponent_team_schedule"]).sum(),
        },
        {
            "source": "mapped snap counts",
            "rows": len(snap_schedule_check),
            "missing schedule rows": snap_schedule_check["schedule_join"].ne("both").sum(),
            "season mismatches": snap_schedule_check["season_source"].ne(snap_schedule_check["season_schedule"]).sum(),
            "week mismatches": snap_schedule_check["week_source"].ne(snap_schedule_check["week_schedule"]).sum(),
            "opponent mismatches": snap_schedule_check["opponent"].ne(snap_schedule_check["opponent_team"]).sum(),
        },
    ]
)

assert source_schedule_summary.iloc[:, 2:].eq(0).all().all()

source_schedule_summary

,source,rows,missing schedule rows,season mismatches,week mismatches,opponent mismatches
0,identified weekly stats,56982,0,0,0,0
1,mapped snap counts,79656,0,0,0,0


### 9. Build the shared player-game observation union

The outer join preserves stats-only and snap-only observations. The explicit one-to-one validation prevents source duplication.

In [11]:
player_game_observations = prepared_stats.merge(
    prepared_snaps,
    on=SOURCE_GAME_KEY,
    how="outer",
    validate="one_to_one",
    indicator="source_join",
)

player_game_observations["has_stat_record"] = (
    player_game_observations["has_stat_record"].eq(True)
)
player_game_observations["has_snap_record"] = (
    player_game_observations["has_snap_record"].eq(True)
)
player_game_observations["participated_game"] = (
    player_game_observations["participated_game"].astype("boolean")
)
player_game_observations["offensive_participant"] = (
    player_game_observations["offensive_participant"].astype("boolean")
)

player_game_observations["source_membership"] = (
    player_game_observations["source_join"]
    .map({"left_only": "stats_only", "right_only": "snaps_only", "both": "both"})
    .astype("string")
)

source_union_summary = (
    player_game_observations["source_membership"]
    .value_counts()
    .rename_axis("source_membership")
    .reset_index(name="rows")
)

assert not player_game_observations.duplicated(SOURCE_GAME_KEY).any()
assert len(player_game_observations) == len(prepared_stats) + len(prepared_snaps) - player_game_observations["source_join"].eq("both").sum()

source_union_summary

,source_membership,rows
0,both,56930
1,snaps_only,22726
2,stats_only,52


### 10. Enrich observations without changing their grain

Schedule context is required. Weekly roster and player-master fields are enrichment; their match flags remain visible. Display-name fallback is explicit even though the current canonical population matches the player master completely.

In [12]:
roster_enrichment = rosters.loc[
    rosters["gsis_id"].notna(),
    [
        "gsis_id",
        "season",
        "week",
        "team",
        "full_name",
        "position",
        "status",
        "status_description_abbr",
    ],
].rename(
    columns={
        "gsis_id": "player_id",
        "full_name": "roster_player_name",
        "position": "roster_position",
        "status": "roster_status",
        "status_description_abbr": "roster_status_description",
    }
)

assert not roster_enrichment.duplicated(WEEKLY_GRAIN).any()

enriched_player_games = player_game_observations.merge(
    schedule_team_games,
    on=["game_id", "team"],
    how="left",
    validate="many_to_one",
    indicator="schedule_join",
).merge(
    roster_enrichment,
    on=WEEKLY_GRAIN,
    how="left",
    validate="one_to_one",
    indicator="roster_join",
).merge(
    player_dimension,
    on="player_id",
    how="left",
    validate="many_to_one",
    indicator="player_master_join",
)

assert len(enriched_player_games) == len(player_game_observations)
assert enriched_player_games["schedule_join"].eq("both").all()

enriched_player_games["has_player_master_record"] = (
    enriched_player_games["player_master_join"].eq("both")
)
enriched_player_games["has_roster_record"] = (
    enriched_player_games["roster_join"].eq("both")
)
enriched_player_games["display_name"] = (
    enriched_player_games["display_name"]
    .combine_first(enriched_player_games["stats_player_name"])
    .combine_first(enriched_player_games["snap_player_name"])
    .combine_first(enriched_player_games["roster_player_name"])
)
enriched_player_games["is_wr_week"] = (
    enriched_player_games["stats_position"].eq("WR")
    | enriched_player_games["snap_position"].eq("WR")
)

enrichment_summary = pd.DataFrame(
    {
        "check": [
            "output observations",
            "schedule matches",
            "player-master matches",
            "weekly-roster matches",
            "contextual WR observations",
        ],
        "rows": [
            len(enriched_player_games),
            enriched_player_games["schedule_join"].eq("both").sum(),
            enriched_player_games["has_player_master_record"].sum(),
            enriched_player_games["has_roster_record"].sum(),
            enriched_player_games["is_wr_week"].sum(),
        ],
    }
)

enrichment_summary

,check,rows
0,output observations,79708
1,schedule matches,79708
2,player-master matches,79708
3,weekly-roster matches,79701
4,contextual WR observations,8881


In [13]:
roster_enrichment_exceptions = enriched_player_games.loc[
    ~enriched_player_games["has_roster_record"],
    [
        "player_id", "display_name", "season", "week", "game_id", "team",
        "stats_position", "snap_position", "has_stat_record", "has_snap_record",
    ],
].sort_values(["season", "week", "player_id"])

print(f"Observed player-games without exact weekly-roster enrichment: {len(roster_enrichment_exceptions):,}")
roster_enrichment_exceptions

Observed player-games without exact weekly-roster enrichment: 7


,player_id,display_name,season,week,game_id,team,stats_position,snap_position,has_stat_record,has_snap_record
0,00-0020895,Chris Smith,2024,1,2024_01_LA_DET,DET,NaN,DT,False,True
1,00-0020895,Chris Smith,2024,9,2024_09_DET_GB,DET,NaN,DT,False,True
70493,00-0039383,Jaylin Simpson,2024,14,2024_14_NYJ_MIA,NYJ,NaN,G,False,True
2,00-0020895,Chris Smith,2024,16,2024_16_DET_CHI,DET,NaN,DT,False,True
3,00-0020895,Chris Smith,2024,17,2024_17_DET_SF,DET,NaN,DT,False,True
4,00-0020895,Chris Smith,2024,18,2024_18_MIN_DET,DET,NaN,DT,False,True
5,00-0020895,Chris Smith,2024,20,2024_20_WAS_DET,DET,NaN,DT,False,True


### 11. Create `silver_player_game_participation`

Grain: one row per `player_id + game_id`. Team remains an attribute and is explicitly tested for uniqueness within the game grain.

In [14]:
PARTICIPATION_COLUMNS = [
    "player_id",
    "game_id",
    "season",
    "week",
    "game_type",
    "season_type",
    "team",
    "opponent_team",
    "display_name",
    "has_player_master_record",
    "has_roster_record",
    "master_position",
    "master_position_group",
    "roster_position",
    "roster_status",
    "roster_status_description",
    "stats_position",
    "stats_position_group",
    "snap_position",
    "is_wr_week",
    "has_stat_record",
    "has_snap_record",
    "participated_game",
    "offensive_participant",
    "offense_snaps",
    "defense_snaps",
    "special_teams_snaps",
    "offense_snap_pct",
    "defense_snap_pct",
    "special_teams_snap_pct",
]

silver_player_game_participation = (
    enriched_player_games[PARTICIPATION_COLUMNS]
    .sort_values(["season", "week", "game_id", "team", "player_id"])
    .reset_index(drop=True)
)

assert silver_player_game_participation[GAME_GRAIN].notna().all().all()
assert not silver_player_game_participation.duplicated(GAME_GRAIN).any()
assert silver_player_game_participation.groupby(GAME_GRAIN)["team"].nunique().le(1).all()

participation_grain_summary = pd.DataFrame(
    {
        "check": [
            "rows",
            "duplicate game-grain rows",
            "null game-grain rows",
            "game-grain keys with multiple teams",
        ],
        "value": [
            len(silver_player_game_participation),
            silver_player_game_participation.duplicated(GAME_GRAIN).sum(),
            silver_player_game_participation[GAME_GRAIN].isna().any(axis=1).sum(),
            silver_player_game_participation.groupby(GAME_GRAIN)["team"].nunique().gt(1).sum(),
        ],
    }
)
participation_grain_summary

,check,value
0,rows,79708
1,duplicate game-grain rows,0
2,null game-grain rows,0
3,game-grain keys with multiple teams,0


### 12. Create `silver_player_week`

The selected seasons contain at most one game per team-week. Weekly facts therefore retain `game_id` as an attribute while enforcing `player_id + season + week + team`.

In [15]:
PLAYER_WEEK_COLUMNS = [
    *PARTICIPATION_COLUMNS,
    *SILVER_STAT_COLUMNS,
]

silver_player_week = (
    enriched_player_games[PLAYER_WEEK_COLUMNS]
    .sort_values(["season", "week", "team", "player_id"])
    .reset_index(drop=True)
)

weekly_game_counts = (
    silver_player_week.groupby(WEEKLY_GRAIN, dropna=False)["game_id"]
    .nunique(dropna=False)
)
weekly_team_counts = (
    silver_player_week.groupby(["player_id", "season", "week"], dropna=False)["team"]
    .nunique(dropna=False)
)

assert silver_player_week[WEEKLY_GRAIN].notna().all().all()
assert not silver_player_week.duplicated(WEEKLY_GRAIN).any()
assert weekly_game_counts.le(1).all()
assert weekly_team_counts.le(1).all()

weekly_grain_summary = pd.DataFrame(
    {
        "check": [
            "rows",
            "duplicate weekly-grain rows",
            "null weekly-grain rows",
            "weekly keys with multiple games",
            "player-weeks with multiple teams",
        ],
        "value": [
            len(silver_player_week),
            silver_player_week.duplicated(WEEKLY_GRAIN).sum(),
            silver_player_week[WEEKLY_GRAIN].isna().any(axis=1).sum(),
            weekly_game_counts.gt(1).sum(),
            weekly_team_counts.gt(1).sum(),
        ],
    }
)
weekly_grain_summary

,check,value
0,rows,79708
1,duplicate weekly-grain rows,0
2,null weekly-grain rows,0
3,weekly keys with multiple games,0
4,player-weeks with multiple teams,0


### 13. Validate null-versus-zero participation behavior

Stats-only records retain null snap fields and null participation. Mapped snap rows retain observed zeroes and Boolean participation results.

In [16]:
participation_null_summary = pd.DataFrame(
    [
        {
            "population": "stats only",
            "rows": silver_player_game_participation["has_stat_record"].eq(True).mul(
                silver_player_game_participation["has_snap_record"].eq(False)
            ).sum(),
            "null participated_game": silver_player_game_participation.loc[
                silver_player_game_participation["has_stat_record"]
                & ~silver_player_game_participation["has_snap_record"],
                "participated_game",
            ].isna().sum(),
        },
        {
            "population": "has mapped snap record",
            "rows": silver_player_game_participation["has_snap_record"].sum(),
            "null participated_game": silver_player_game_participation.loc[
                silver_player_game_participation["has_snap_record"],
                "participated_game",
            ].isna().sum(),
        },
    ]
)

stats_only_mask = (
    silver_player_game_participation["has_stat_record"]
    & ~silver_player_game_participation["has_snap_record"]
)
assert silver_player_game_participation.loc[stats_only_mask, "participated_game"].isna().all()
assert silver_player_game_participation.loc[stats_only_mask, "offensive_participant"].isna().all()
assert silver_player_game_participation.loc[~silver_player_game_participation["has_snap_record"], ["offense_snaps", "defense_snaps", "special_teams_snaps"]].isna().all().all()
assert silver_player_game_participation.loc[silver_player_game_participation["has_snap_record"], "participated_game"].notna().all()

participation_null_summary

,population,rows,null participated_game
0,stats only,52,52
1,has mapped snap record,79656,0


### 14. Reconcile Silver statistics and snaps to Bronze

Every selected weekly-stat total must reconcile exactly. Snap totals reconcile to the mapped source population; unresolved PFR identities remain outside canonical output by design.

In [17]:
stat_reconciliation_records = []
for source_column, silver_column in STAT_COLUMN_RENAMES.items():
    for season in SEASONS:
        bronze_total = identified_player_stats.loc[
            identified_player_stats["season"] == season, source_column
        ].sum()
        silver_total = silver_player_week.loc[
            silver_player_week["season"] == season, silver_column
        ].sum()
        stat_reconciliation_records.append(
            {
                "season": season,
                "source_column": source_column,
                "silver_column": silver_column,
                "bronze_total": bronze_total,
                "silver_total": silver_total,
                "difference": silver_total - bronze_total,
            }
        )

stat_reconciliation = pd.DataFrame(stat_reconciliation_records)
assert stat_reconciliation["difference"].eq(0).all()

stat_reconciliation_summary = (
    stat_reconciliation.groupby("season")
    .agg(
        metrics_checked=("silver_column", "size"),
        mismatched_metrics=("difference", lambda values: values.ne(0).sum()),
        largest_absolute_difference=("difference", lambda values: values.abs().max()),
    )
    .reset_index()
)
stat_reconciliation_summary

,season,metrics_checked,mismatched_metrics,largest_absolute_difference
0,2023,24,0,0
1,2024,24,0,0
2,2025,24,0,0


In [18]:
SNAP_COLUMN_PAIRS = {
    "offense_snaps": "offense_snaps",
    "defense_snaps": "defense_snaps",
    "st_snaps": "special_teams_snaps",
}

snap_reconciliation_records = []
for source_column, silver_column in SNAP_COLUMN_PAIRS.items():
    for season in SEASONS:
        bronze_total = mapped_snap_source.loc[
            mapped_snap_source["season"] == season, source_column
        ].sum()
        silver_total = silver_player_game_participation.loc[
            (silver_player_game_participation["season"] == season)
            & silver_player_game_participation["has_snap_record"],
            silver_column,
        ].sum()
        snap_reconciliation_records.append(
            {
                "season": season,
                "source_column": source_column,
                "silver_column": silver_column,
                "bronze_total": bronze_total,
                "silver_total": silver_total,
                "difference": silver_total - bronze_total,
            }
        )

snap_reconciliation = pd.DataFrame(snap_reconciliation_records)
assert snap_reconciliation["difference"].eq(0).all()

snap_reconciliation

,season,source_column,silver_column,bronze_total,silver_total,difference
0,2023,offense_snaps,offense_snaps,413342.0,413342,0.0
1,2024,offense_snaps,offense_snaps,409253.0,409253,0.0
2,2025,offense_snaps,offense_snaps,403189.0,403189,0.0
3,2023,defense_snaps,defense_snaps,413398.0,413398,0.0
4,2024,defense_snaps,defense_snaps,409736.0,409736,0.0
5,2025,defense_snaps,defense_snaps,403097.0,403097,0.0
6,2023,st_snaps,special_teams_snaps,167395.0,167395,0.0
7,2024,st_snaps,special_teams_snaps,167407.0,167407,0.0
8,2025,st_snaps,special_teams_snaps,164501.0,164501,0.0


### 15. Reproduce the WR validation baseline

The contextual WR slice must reproduce the 8,881-row union from the preceding notebook. Position conflicts remain source-specific rather than being overwritten.

In [19]:
silver_wr_weeks = silver_player_week.loc[silver_player_week["is_wr_week"]].copy()

wr_position_conflicts = silver_wr_weeks.loc[
    silver_wr_weeks["stats_position"].eq("WR")
    & silver_wr_weeks["roster_position"].notna()
    & silver_wr_weeks["roster_position"].ne("WR")
].copy()

wr_baseline_summary = pd.DataFrame(
    {
        "check": [
            "contextual WR player-weeks",
            "distinct WR players",
            "weekly-stat rows labeled WR",
            "mapped snap rows labeled WR",
            "stats-WR rows with non-WR roster position",
        ],
        "value": [
            len(silver_wr_weeks),
            silver_wr_weeks["player_id"].nunique(),
            silver_player_week["stats_position"].eq("WR").sum(),
            silver_player_week["snap_position"].eq("WR").sum(),
            len(wr_position_conflicts),
        ],
    }
)

assert len(silver_wr_weeks) == 8881
assert silver_player_week["stats_position"].eq("WR").sum() == 7795
assert silver_player_week["snap_position"].eq("WR").sum() == 8848
assert len(wr_position_conflicts) == 23

wr_baseline_summary

,check,value
0,contextual WR player-weeks,8881
1,distinct WR players,371
2,weekly-stat rows labeled WR,7795
3,mapped snap rows labeled WR,8848
4,stats-WR rows with non-WR roster position,23


In [20]:
special_teams_only_wr = silver_wr_weeks.loc[
    silver_wr_weeks["has_snap_record"]
    & silver_wr_weeks["participated_game"].eq(True)
    & silver_wr_weeks["offensive_participant"].eq(False),
    [
        "player_id", "display_name", "season", "week", "team",
        "stats_position", "snap_position", "offense_snaps", "defense_snaps",
        "special_teams_snaps",
    ],
].sort_values(["season", "week", "team", "display_name"])

print(f"Contextual WR rows with non-offensive participation only: {len(special_teams_only_wr):,}")
special_teams_only_wr.head(20)

Contextual WR rows with non-offensive participation only: 453


,player_id,display_name,season,week,team,stats_position,snap_position,offense_snaps,defense_snaps,special_teams_snaps
23,00-0035500,Greg Dortch,2023,1,ARI,WR,WR,0,0,11
125,00-0036630,Tylan Wallace,2023,1,BAL,NaN,WR,0,0,16
320,00-0038576,Charlie Jones,2023,1,CIN,WR,WR,0,0,8
940,00-0034646,Brandon Powell,2023,1,MIN,WR,WR,0,0,6
961,00-0037291,Jalen Nailor,2023,1,MIN,WR,WR,0,0,18
974,00-0026293,Matthew Slater,2023,1,NE,NaN,WR,0,0,27
1140,00-0035140,Mecole Hardman,2023,1,NYJ,NaN,WR,0,0,1
1202,00-0037132,Britain Covey,2023,1,PHI,WR,WR,0,0,5
1437,00-0038969,Kearis Jackson,2023,1,TEN,WR,WR,0,0,10
1452,00-0034297,Byron Pringle,2023,1,WAS,NaN,WR,0,0,1


### 16. Check recognizable team changes

Team changes across weeks are valid; same-week multi-team rows would violate the current weekly contract.

In [21]:
player_season_team_counts = (
    silver_player_week.groupby(["player_id", "season"], dropna=False)["team"]
    .nunique(dropna=False)
    .reset_index(name="team_count")
)
weekly_team_counts = (
    silver_player_week.groupby(["player_id", "season", "week"], dropna=False)["team"]
    .nunique(dropna=False)
    .reset_index(name="team_count")
)

assert not weekly_team_counts["team_count"].gt(1).any()

recognizable_players = [
    "Amari Cooper",
    "Davante Adams",
    "DeAndre Hopkins",
    "Mecole Hardman",
    "Donovan Peoples-Jones",
]
trade_examples = silver_player_week.loc[
    silver_player_week["display_name"].isin(recognizable_players)
    & silver_player_week["is_wr_week"],
    ["player_id", "display_name", "season", "week", "team", "game_id", "participated_game"],
].sort_values(["display_name", "season", "week"])

print(
    "All-position player-seasons with multiple observed teams: "
    f"{player_season_team_counts['team_count'].gt(1).sum():,}"
)
print(
    "Same-week multi-team player observations: "
    f"{weekly_team_counts['team_count'].gt(1).sum():,}"
)
trade_examples.head(40)

All-position player-seasons with multiple observed teams: 302
Same-week multi-team player observations: 0


,player_id,display_name,season,week,team,game_id,participated_game
329,00-0031544,Amari Cooper,2023,1,CLE,2023_01_CIN_CLE,True
1817,00-0031544,Amari Cooper,2023,2,CLE,2023_02_CLE_PIT,True
3295,00-0031544,Amari Cooper,2023,3,CLE,2023_03_TEN_CLE,True
4786,00-0031544,Amari Cooper,2023,4,CLE,2023_04_BAL_CLE,True
7586,00-0031544,Amari Cooper,2023,6,CLE,2023_06_SF_CLE,True
8887,00-0031544,Amari Cooper,2023,7,CLE,2023_07_CLE_IND,True
10183,00-0031544,Amari Cooper,2023,8,CLE,2023_08_CLE_SEA,True
11675,00-0031544,Amari Cooper,2023,9,CLE,2023_09_ARI_CLE,True
12980,00-0031544,Amari Cooper,2023,10,CLE,2023_10_CLE_BAL,True
14239,00-0031544,Amari Cooper,2023,11,CLE,2023_11_PIT_CLE,True


### 17. Persist season-partitioned Silver Parquet

Each partition is written and immediately reloaded. Shape, columns, dtypes, season membership, and declared grain must survive the round trip.

In [22]:
GAME_PARTICIPATION_DIR.mkdir(parents=True, exist_ok=True)
PLAYER_WEEK_DIR.mkdir(parents=True, exist_ok=True)

output_records = []

for season in SEASONS:
    participation_path = (
        GAME_PARTICIPATION_DIR / f"player_game_participation_{season}.parquet"
    )
    participation_partition = (
        silver_player_game_participation.loc[
            silver_player_game_participation["season"] == season
        ].reset_index(drop=True)
    )
    participation_partition.to_parquet(participation_path, index=False)
    participation_reloaded = pd.read_parquet(participation_path)

    assert participation_reloaded.shape == participation_partition.shape
    assert participation_reloaded.columns.tolist() == participation_partition.columns.tolist()
    assert participation_reloaded.dtypes.astype(str).tolist() == participation_partition.dtypes.astype(str).tolist()
    assert set(participation_reloaded["season"].unique()) == {season}
    assert not participation_reloaded.duplicated(GAME_GRAIN).any()

    output_records.append(
        {
            "dataset": "player_game_participation",
            "season": season,
            "rows": len(participation_reloaded),
            "columns": len(participation_reloaded.columns),
            "grain": " + ".join(GAME_GRAIN),
            "duplicate_grain_rows": participation_reloaded.duplicated(GAME_GRAIN).sum(),
            "file_mb": round(participation_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

    player_week_path = PLAYER_WEEK_DIR / f"player_week_{season}.parquet"
    player_week_partition = silver_player_week.loc[
        silver_player_week["season"] == season
    ].reset_index(drop=True)
    player_week_partition.to_parquet(player_week_path, index=False)
    player_week_reloaded = pd.read_parquet(player_week_path)

    assert player_week_reloaded.shape == player_week_partition.shape
    assert player_week_reloaded.columns.tolist() == player_week_partition.columns.tolist()
    assert player_week_reloaded.dtypes.astype(str).tolist() == player_week_partition.dtypes.astype(str).tolist()
    assert set(player_week_reloaded["season"].unique()) == {season}
    assert not player_week_reloaded.duplicated(WEEKLY_GRAIN).any()

    output_records.append(
        {
            "dataset": "player_week",
            "season": season,
            "rows": len(player_week_reloaded),
            "columns": len(player_week_reloaded.columns),
            "grain": " + ".join(WEEKLY_GRAIN),
            "duplicate_grain_rows": player_week_reloaded.duplicated(WEEKLY_GRAIN).sum(),
            "file_mb": round(player_week_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

output_summary = pd.DataFrame(output_records).sort_values(["dataset", "season"]).reset_index(drop=True)
output_summary

,dataset,season,rows,columns,grain,duplicate_grain_rows,file_mb,round_trip_passed
0,player_game_participation,2023,26532,30,player_id + game_id,0,0.34,True
1,player_game_participation,2024,26587,30,player_id + game_id,0,0.34,True
2,player_game_participation,2025,26589,30,player_id + game_id,0,0.34,True
3,player_week,2023,26532,54,player_id + season + week + team,0,0.53,True
4,player_week,2024,26587,54,player_id + season + week + team,0,0.54,True
5,player_week,2025,26589,54,player_id + season + week + team,0,0.54,True


In [23]:
print(f"Silver files written: {len(output_summary):,}")
print(f"Rows written across partitions: {output_summary['rows'].sum():,}")
print(f"Total Parquet size: {output_summary['file_mb'].sum():,.2f} MB")
print("All round trips passed:", output_summary["round_trip_passed"].all())

assert len(output_summary) == 6
assert output_summary["round_trip_passed"].all()
assert output_summary["duplicate_grain_rows"].eq(0).all()

Silver files written: 6
Rows written across partitions: 159,416
Total Parquet size: 2.63 MB
All round trips passed: True


## Takeaways

- The shared stats-plus-mapped-snaps population produces 79,708 canonical player-game and player-week facts without join multiplication.
- Weekly stats alone are participation-incomplete: 22,726 mapped snap observations have no weekly-stat row. Conversely, 52 stats rows have no mapped snap record and correctly retain unknown participation rather than false participation.
- Five mapped snap rows explicitly record zero total snaps; these are observed non-participation and remain distinct from missing snap evidence.
- All output game/team relationships match schedules. Seven facts lack exact weekly-roster enrichment, so roster status and roster position remain nullable context rather than required keys.
- All 24 scoring-neutral weekly metrics and the three snap-count totals reconcile exactly to their eligible Bronze populations for every season.
- The contextual WR slice reproduces the previously validated 8,881 rows, including 23 stats-WR rows with a different weekly roster position.
- Eleven unresolved PFR identifiers account for 111 excluded snap rows. They remain visible in the notebook exception tables and are not repaired by name.
- Both Silver datasets are persisted by season and pass schema, dtype, grain, partition, and round-trip checks. The next shared Silver notebook can focus on standardized plays and team-week opportunity denominators.